## Example 1: FlexCache Package Exploration and Basic Concepts

In [ ]:
import flexcache
import tempfile
import os

# Example 1: FlexCache Package Exploration and Basic Concepts
print("Example 1: FlexCache Package Exploration and Basic Concepts")

try:
    print(f"✅ FlexCache v{flexcache.version('flexcache')} imported successfully")
    
    # Show available cache classes
    cache_classes = ['DiskCache', 'DiskCacheByHash', 'DiskCacheByMTime']
    print(f"\nAvailable cache classes:")
    for class_name in cache_classes:
        cls = getattr(flexcache, class_name)
        print(f"  - {class_name}: {cls.__doc__.split('.')[0] if cls.__doc__ else 'Disk-based caching'}")
    
    # Show naming strategies
    naming_strategies = ['NameByObj', 'NameByPath', 'NameByHashIter', 'NameByFields']
    print(f"\nNaming strategies:")
    for strategy_name in naming_strategies:
        strategy = getattr(flexcache, strategy_name)
        print(f"  - {strategy_name}: Content-based naming")
    
    # Show invalidation strategies
    invalidation_strategies = ['InvalidateByPathMTime', 'InvalidateByExist']
    print(f"\nInvalidation strategies:")
    for invalidation_name in invalidation_strategies:
        invalidation = getattr(flexcache, invalidation_name)
        print(f"  - {invalidation_name}: Time/existence-based validation")
    
    # Create a temporary directory to test cache initialization
    with tempfile.TemporaryDirectory() as temp_dir:
        cache = flexcache.DiskCache(temp_dir)
        print(f"\n✅ DiskCache successfully initialized at {temp_dir}")
        print(f"Cache type: {type(cache).__name__}")
        
        # Show cache configuration
        print(f"Cache folder: {cache.cache_folder}")
    
    print("\n✅ FlexCache exploration completed successfully!")

except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"flexcache example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed")

## Example 2: DiskCacheByMTime save/load round-trip, stale once the source file is newer

In [ ]:
import os, pathlib, tempfile, time
import flexcache

# Example 2: DiskCacheByMTime -- an entry is named by the source file's path and goes stale when the file is newer
print("Example 2: DiskCacheByMTime save/load round-trip and invalidation by modification time")

folder = pathlib.Path(tempfile.mkdtemp(prefix="flexcache_nb_"))
source = folder / "table.csv"
source.write_text("1,2,3\n")
an_hour_ago = time.time() - 3600
os.utime(source, (an_hour_ago, an_hour_ago))  # older than the entry saved next, whatever the clock resolution

cache = flexcache.DiskCacheByMTime(cache_folder=folder / "cache")
parsed = [1, 2, 3]
stem = cache.save(parsed, source)
assert (folder / "cache" / f"{stem}.pickle").is_file() and (folder / "cache" / f"{stem}.json").is_file()
assert cache.load(source) == (parsed, stem), cache.load(source)
assert cache.load(str(source)) == (parsed, stem)  # a str path names the same entry as the pathlib.Path

in_an_hour = time.time() + 3600
os.utime(source, (in_an_hour, in_an_hour))  # touched: same content, but now newer than the cached entry
value, same_stem = cache.load(source)
assert value is None and same_stem == stem, (value, same_stem)
print(f"OK -- {stem[:12]}... loads back {parsed}; after the source is touched the entry is stale and load() returns None")

## Example 3: DiskCacheByHash entries named by file content

In [ ]:
import pathlib, tempfile
import flexcache

# Example 3: DiskCacheByHash -- an entry is named by the bytes of the source file, not by its path
print("Example 3: DiskCacheByHash save/load round-trip and invalidation by content")

folder = pathlib.Path(tempfile.mkdtemp(prefix="flexcache_nb_"))
cache = flexcache.DiskCacheByHash(cache_folder=folder / "cache")
first = folder / "first.txt"
first.write_text("the quick brown fox")
summary = {"words": 4, "first": "the"}
stem = cache.save(summary, first)
assert cache.load(first) == (summary, stem), cache.load(first)

copy = folder / "copy.txt"
copy.write_text("the quick brown fox")
assert cache.load(copy) == (summary, stem)  # same bytes at another path: the same entry

first.write_text("the quick brown fox jumps")  # new content, so a new name and no entry yet
value, new_stem = cache.load(first)
assert value is None and new_stem != stem, (value, new_stem)

first.write_text("the quick brown fox")  # back to the old bytes: the old entry is found again
assert cache.load(first) == (summary, stem)
print(f"OK -- one entry ({stem[:12]}...) serves both files with the same bytes; changing the bytes misses")

## Example 4: load() with a converter: convert on a miss, reuse on a hit

In [ ]:
import pathlib, tempfile
import flexcache

# Example 4: load() with a converter -- a miss runs it and stores the result, a hit does not run it
print("Example 4: load() with a converter function")

folder = pathlib.Path(tempfile.mkdtemp(prefix="flexcache_nb_"))
cache = flexcache.DiskCacheByHash(cache_folder=folder / "cache")
source = folder / "notes.txt"
source.write_text("alpha beta gamma")

calls = []
def word_count(path):
    calls.append(path)
    return {"words": len(path.read_text().split())}

value, stem = cache.load(source, word_count)  # miss: converts and stores
assert value == {"words": 3} and len(calls) == 1, (value, calls)
assert cache.load(source, word_count) == (value, stem) and len(calls) == 1  # hit: the converter is not called
assert cache.load(source, "word_count") == (value, stem)  # the converter's name alone finds the entry
assert cache.load(source)[0] is None  # the converter's name is part of the entry's name

source.write_text("alpha beta gamma delta")
value, new_stem = cache.load(source, word_count)  # new content: converted again
assert value == {"words": 4} and new_stem != stem and len(calls) == 2, (value, calls)

seen = []
def with_stem(path, cached_stem):
    seen.append(cached_stem)
    return path.read_text().upper()
upper, upper_stem = cache.load(source, with_stem, pass_hash=True)  # the converter is also given the entry's name
assert upper == "ALPHA BETA GAMMA DELTA" and seen == [upper_stem], (upper, seen, upper_stem)
print(f"OK -- word_count ran only on the {len(calls)} misses; pass_hash=True hands the converter the entry name")

## Example 5: DiskCache with a registered header class for non-file sources

In [ ]:
import pathlib, tempfile
from dataclasses import dataclass
import flexcache

# Example 5: a plain DiskCache with a header class for non-file sources
print("Example 5: DiskCache with a registered header class")

@dataclass(frozen=True)
class ParamsHeader(flexcache.NameByObj, flexcache.InvalidateByExist, flexcache.BaseHeader):
    """Names an entry by pickling the source object; the entry stays valid while its file exists."""

cache = flexcache.DiskCache(cache_folder=pathlib.Path(tempfile.mkdtemp(prefix="flexcache_nb_")))
try:
    cache.load(("resnet", 50))
except TypeError:
    pass  # no header class is registered for tuples yet
else:
    raise AssertionError("DiskCache loaded a tuple source with no header class registered")

cache.register_header_class(tuple, ParamsHeader)
result = {"accuracy": 0.76}
stem = cache.save(result, ("resnet", 50), "train")
assert cache.load(("resnet", 50), "train") == (result, stem)
assert cache.load(("resnet", 101), "train")[0] is None  # other parameters, other entry
(cache.cache_folder / f"{stem}.pickle").unlink()
assert cache.load(("resnet", 50), "train")[0] is None  # InvalidateByExist: gone once the file is removed
print("OK -- a tuple-keyed entry round-trips, other parameters miss, and removing the file invalidates it")